# Stop Before It Memorizes

**Topic:** Early stopping &nbsp;·&nbsp; **Difficulty:** Medium

Early stopping is regularization you get for free: watch a validation score, keep
the weights from the best epoch, and quit when it stops improving. The subtlety
is that "quit" and "keep the best" are different things: a model that stops at
epoch 60 but returns the epoch-60 weights is worse than one that returns the
epoch-31 weights it already had.

Train a network that clearly overfits, and implement patience with weight
restoration.

## The data (created in the next cell)

- `X_train` `(120, 40)`, `y_train`, `X_val` `(200, 40)`, `y_val`,
  `X_test` `(400, 40)`, `y_test`.
- Tiny training set, wide model, noisy target: overfitting is guaranteed.
- Train a 40 -> 256 -> 1 tanh network by full-batch gradient descent, 1500
  epochs, lr 0.15, seeded with `np.random.default_rng(13)`.

## Your task

With **NumPy only**:

1. Train for 1500 epochs, recording the validation MSE after each epoch.
   Initialise `W1 ~ N(0, 1/40)` with shape `(40, 256)`, `b1 = zeros(256)`,
   `W2 ~ N(0, 1/256)` with shape `(256, 1)`, `b2 = zeros(1)`, all drawn in that
   order from `np.random.default_rng(13)` before the first epoch, and treat
   validation as improved only when `v < best_val - 1e-12`.
2. Track the best validation MSE and the epoch it occurred at, saving a copy of
   the weights whenever validation improves.
3. Apply patience 150: stop early if validation has not improved for 150
   consecutive epochs. Record `stopped_epoch` (1-based, 1500 if never triggered).
4. Set `answers` to a dict with `"val_curve"` (length up to 1500, one entry per
   epoch actually run), `"best_epoch"` (int, 1-based), `"best_val"`,
   `"test_mse_best"` (test MSE using the restored best weights), and
   `"test_mse_final"` (test MSE using the last weights).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the restored weights beat the final ones on test.

In [ ]:
import numpy as np

def make(n, seed):
    r = np.random.default_rng(seed)
    Z = r.normal(0, 1, (n, 40))
    t = 1.2 * Z[:, 0] - 0.9 * Z[:, 1] + 0.6 * np.tanh(Z[:, 2])
    return Z, t + r.normal(0, 1.1, n)

X_train, y_train = make(120, 81)
X_val, y_val = make(200, 82)
X_test, y_test = make(400, 83)
n_epochs, patience, lr = 1500, 150, 0.15
print(f"train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

_DIGEST_SEEDS = [9176, 5181]

def _digest(a, seed=9176):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(seed).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("val_curve", [162], 270.06409595464913,
     [4.3102679803663975, -0.78041511752952064], 1e-06),
    ("best_epoch", [], 12.0, -2.00925617777399, 1e-06),
    ("best_val", [], 1.461088602512451, -0.24464177507277565, 1e-06),
    ("test_mse_best", [], 2.125274593778383, -0.3558517589179436, 1e-06),
    ("test_mse_final", [], 2.2519475119413013, -0.3770616208659021, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _vals = _probe if isinstance(_probe, list) else [_probe]
    _g = abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
    for _seed, _pv in zip(_DIGEST_SEEDS, _vals):
        _g = (_g and abs(_digest(_a, _seed) - _pv)
              <= _tol * (1.0 + abs(_pv)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "test_mse_best" in answers:
    _g = answers["test_mse_best"] <= answers["test_mse_final"]
    print(f"  {'restoring beats final':<24} {'ok' if _g else 'wrong'}  "
          f"({answers['test_mse_best']:.4f} vs {answers['test_mse_final']:.4f})")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet, revisit your solution and try again.")

## Solution

The validation curve falls, bottoms out, and climbs as the network starts fitting
the training noise, and patience just detects that the climb is real rather than a
blip. The important half is restoration: without it you return whatever the model
looked like when you happened to stop, which is by construction worse than the
best point you already saw. Keeping a copy of the weights costs a copy of each
parameter array and is what `restore_best_weights=True` does in Keras.

```python
import numpy as np

rng = np.random.default_rng(13)
d, h = 40, 256
W1 = rng.normal(0, np.sqrt(1.0 / d), (d, h))
b1 = np.zeros(h)
W2 = rng.normal(0, np.sqrt(1.0 / h), (h, 1))
b2 = np.zeros(1)

Y = y_train.reshape(-1, 1)
n = len(y_train)

def predict(Xa, P):
    w1, c1, w2, c2 = P
    return (np.tanh(Xa @ w1 + c1) @ w2 + c2).ravel()

def mse(p, t):
    return float(np.mean((p - t) ** 2))

best_val = np.inf
best_epoch = 0
best_params = None
since = 0
val_curve = []
stopped_epoch = n_epochs

for ep in range(1, n_epochs + 1):
    H = np.tanh(X_train @ W1 + b1)
    P = H @ W2 + b2
    dP = 2.0 * (P - Y) / n
    gW2 = H.T @ dP
    gb2 = dP.sum(axis=0)
    dA = (dP @ W2.T) * (1 - H ** 2)
    gW1 = X_train.T @ dA
    gb1 = dA.sum(axis=0)
    W1 -= lr * gW1
    b1 -= lr * gb1
    W2 -= lr * gW2
    b2 -= lr * gb2

    v = mse(predict(X_val, (W1, b1, W2, b2)), y_val)
    val_curve.append(v)
    if v < best_val - 1e-12:
        best_val, best_epoch = v, ep
        best_params = (W1.copy(), b1.copy(), W2.copy(), b2.copy())
        since = 0
    else:
        since += 1
        if since >= patience:
            stopped_epoch = ep
            break

answers = {
    "val_curve": np.array(val_curve),
    "best_epoch": int(best_epoch),
    "best_val": float(best_val),
    "test_mse_best": mse(predict(X_test, best_params), y_test),
    "test_mse_final": mse(predict(X_test, (W1, b1, W2, b2)), y_test),
}
```